# CoT-Trace Vis-Head Discovery (Qwen3-VL-8B-Thinking)

Every discovery method used elsewhere in this project measures attention at a
**single query position**: the last prompt token, right before generation begins.
For a reasoning/CoT-tuned model like `Qwen3-VL-8B-Thinking`, that position comes
*before* any of the actual reasoning happens — the model hasn't started "looking
back" at the image yet in the way it will while working through its
chain-of-thought. This may be why the standard method found unremarkable,
Instruct-like discovery scores for Thinking, yet steering those same top-ranked
heads had **no significant causal effect** (see `vis_head_across_qwen3vl_stages.ipynb`).

This notebook implements a CoT-aware alternative:

```
Multimodal CoT sample -> Qwen3-VL-Thinking generates its full reasoning trace
    -> at EVERY decode step during that generation, record each head's
       attention back to the image tokens (register_attention_trackers)
    -> restrict to the ground-truth target region's tokens
    -> average target-region attention mass across all decode steps
       ("CoT -> visual attention", object-specific)
    -> normalize by the target region's token-count share of the image
       ("area normalization" -- controls for region size)
    -> aggregate across many samples -> rank heads -> candidate CoT heads
    -> causal intervention (steer the candidate heads) -> confirm
```

The candidate ranking is validated the same way as every other ranking in this
project: causal steering on a held-out MCQ set, compared directly against the
standard (final-query-only) ranking on the *same* eval samples, so any
difference in steered accuracy is attributable to the ranking method alone.

In [1]:
%matplotlib inline
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

from vis_head.common import DEFAULT_SEED
from vis_head.gaze import aggregate_region_attention, collect_cot_trace_region_attention, collect_last_query_attentions, rank_heads_by_score
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, PROMPT_TEMPLATES, mcq_prompt, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import (group_heads_by_layer, intervention_positions, make_static_attention_mask_hook,
                                register_mask_hooks, remove_handles)

MODEL_ID = "Qwen/Qwen3-VL-8B-Thinking"
DEVICE = "cuda:0"
SEED = DEFAULT_SEED
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_DISCOVERY_SAMPLES = 150       # full CoT generation per sample is far more expensive than one forward pass
COT_DISCOVERY_MAX_NEW_TOKENS = 150   # enough to capture the reasoning-about-the-image phase (confirmed via diagnostic: model identifies objects well within this budget)
TOP_K_HEADS = 15
N_MCQ_SAMPLES = 150
MCQ_MAX_NEW_TOKENS = 600        # Thinking needs a full trace to reach the answer letter (see vis_head_across_qwen3vl_stages.ipynb)
N_OPTIONS = 4
OPTION_LETTERS = ["A", "B", "C", "D"][:N_OPTIONS]

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)

model, processor = load_model_and_processor(model_id=MODEL_ID, device=DEVICE)
n_layers, n_heads, spatial_merge = model_dims(model)
print(f"{n_layers} layers x {n_heads} heads")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


## Method A (existing, for comparison): final-query-only discovery

In [2]:
def discover_vis_head_scores_final_query(prompt_fn, label, seed):
    rng = np.random.RandomState(seed)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(N_DISCOVERY_SAMPLES), desc=f"[{label}] final-query discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = prompt_fn(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            attn_at_query = collect_last_query_attentions(model, inputs)
            region_attention = aggregate_region_attention(attn_at_query=attn_at_query, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            raw_sum += region_attention[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"Skipping grid: {exc}")
    print(f"  [{label}] valid={valid}/{N_DISCOVERY_SAMPLES}")
    return (raw_sum / max(valid, 1)).astype(np.float32)

## Method B (new): CoT-trace discovery

Generates the model's full reasoning trace and averages target-region attention
mass across every decode step, area-normalized by the target region's share of
total image tokens (so a head that merely tracks "some part of the image,
proportional to its size" scores near 1.0 -- meaningfully above-chance
selectivity requires a score above that).

In [3]:
def discover_cot_vis_head_scores(prompt_fn, label, seed):
    """Thin per-sample wrapper around vis_head.gaze.collect_cot_trace_region_attention,
    aggregating across many discovery samples."""
    rng = np.random.RandomState(seed)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(N_DISCOVERY_SAMPLES), desc=f"[{label}] CoT-trace discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                            class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = prompt_fn(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs(processor, grid.grid, prompt, DEVICE)
            region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
            img_start, img_end = find_image_token_range(inputs, processor)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[target_cell]
            scores = collect_cot_trace_region_attention(
                model=model, inputs=inputs, target_positions=target_positions,
                img_start=img_start, img_end=img_end, n_layers=n_layers, n_heads=n_heads,
                max_new_tokens=COT_DISCOVERY_MAX_NEW_TOKENS,
            )
            raw_sum += scores
            valid += 1
        except Exception as exc:
            print(f"Skipping grid: {exc}")
    print(f"  [{label}] valid={valid}/{N_DISCOVERY_SAMPLES}")
    return (raw_sum / max(valid, 1)).astype(np.float32)

## Run both discovery methods

In [4]:
prompt_fn = lambda name: PROMPT_TEMPLATES["find"].format(name=name)

scores_final_query = discover_vis_head_scores_final_query(prompt_fn, "final_query", seed=SEED + 100)
ranked_final_query = rank_heads_by_score(scores_final_query)
print(f"Method A (final-query) mean score: {scores_final_query.mean():.5f}")
print("Top-10:", [(r["layer"], r["head"]) for r in ranked_final_query[:10]])

scores_cot = discover_cot_vis_head_scores(prompt_fn, "cot_trace", seed=SEED + 200)
ranked_cot = rank_heads_by_score(scores_cot)
print(f"\nMethod B (CoT-trace) mean score: {scores_cot.mean():.5f}  (area-normalized -- 1.0 = chance)")
print("Top-10:", [(r["layer"], r["head"]) for r in ranked_cot[:10]])

top15_a = set((r["layer"], r["head"]) for r in ranked_final_query[:TOP_K_HEADS])
top15_b = set((r["layer"], r["head"]) for r in ranked_cot[:TOP_K_HEADS])
overlap = top15_a & top15_b
print(f"\nTop-{TOP_K_HEADS} overlap between methods: {len(overlap)}/{TOP_K_HEADS}  {sorted(overlap)}")

[final_query] final-query discovery:   0%|          | 0/150 [00:00<?, ?it/s]

  [final_query] valid=150/150
Method A (final-query) mean score: 0.02287
Top-10: [(0, 27), (2, 17), (0, 8), (0, 5), (0, 29), (5, 2), (1, 6), (10, 24), (3, 20), (26, 19)]


[cot_trace] CoT-trace discovery:   0%|          | 0/150 [00:00<?, ?it/s]

  [cot_trace] valid=150/150

Method B (CoT-trace) mean score: 0.12484  (area-normalized -- 1.0 = chance)
Top-10: [(20, 15), (15, 11), (17, 27), (17, 4), (18, 15), (26, 18), (21, 8), (21, 10), (21, 11), (17, 24)]

Top-15 overlap between methods: 0/15  []


## Causal validation: does the CoT-trace ranking steer better than the final-query ranking?

Same held-out MCQ samples, same no-cue prompt, same steering mechanism -- only
the head *ranking* differs between the two steered conditions.

In [5]:
def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng,
                        class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    n_distractors = min(N_OPTIONS - 1, len(other_names))
    distractor_idx = rng.choice(len(other_names), size=n_distractors, replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    location_prompt = mcq_prompt(target_cell + 1, ROWS, COLS, options, OPTION_LETTERS[:len(options)])
    return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter,
            "prompt": prompt, "location_prompt": location_prompt}


def extract_letter(text, valid_letters):
    matches = re.findall(r"\b([" + "".join(valid_letters) + r"])\b", text.upper())
    return matches[-1] if matches else None   # last mention -- CoT text discusses multiple letters before concluding


def run_mcq(sample, heads_by_layer, prompt_key="prompt"):
    inputs = prepare_inputs(processor, sample["grid"].grid, sample[prompt_key], DEVICE)
    prompt_length = int(inputs["input_ids"].shape[1])
    handles = []
    if heads_by_layer is not None:
        img_start, img_end = find_image_token_range(inputs, processor)
        region_ids, _ = assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=ROWS, cols=COLS, spatial_merge=spatial_merge)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[sample["target_cell"]]
        other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                boost_positions=boost_positions, n_query_heads=n_heads,
                                                device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for l, hh in heads_by_layer.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
    try:
        seq = run_generation(model=model, inputs=inputs, max_new_tokens=MCQ_MAX_NEW_TOKENS)
    finally:
        remove_handles(handles)
    text = decode_generated_text(processor, seq, prompt_length)
    valid_letters = OPTION_LETTERS[: len(sample["options"])]
    predicted = extract_letter(text, valid_letters)
    return {"predicted": predicted, "correct": predicted == sample["correct_letter"]}


mcq_rng = np.random.RandomState(SEED + 555)
mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(N_MCQ_SAMPLES)]

heads_final_query = group_heads_by_layer(list(top15_a))
heads_cot = group_heads_by_layer(list(top15_b))

baseline_res, steered_a_res, steered_b_res = [], [], []
for sample in tqdm(mcq_samples, desc="MCQ eval"):
    baseline_res.append(run_mcq(sample, None, "prompt"))
    steered_a_res.append(run_mcq(sample, heads_final_query, "prompt"))
    steered_b_res.append(run_mcq(sample, heads_cot, "prompt"))

b_acc = np.mean([r["correct"] for r in baseline_res])
a_acc = np.mean([r["correct"] for r in steered_a_res])
b2_acc = np.mean([r["correct"] for r in steered_b_res])
print(f"baseline                         : {b_acc:.3f}")
print(f"steered (final-query ranking, A) : {a_acc:.3f}")
print(f"steered (CoT-trace ranking, B)   : {b2_acc:.3f}")

MCQ eval:   0%|          | 0/150 [00:00<?, ?it/s]

baseline                         : 0.280
steered (final-query ranking, A) : 0.240
steered (CoT-trace ranking, B)   : 0.713


## Significance

In [6]:
def mcnemar_p(cond_a, cond_b):
    b = sum(1 for a, c in zip(cond_a, cond_b) if not a["correct"] and c["correct"])
    c = sum(1 for a, c in zip(cond_a, cond_b) if a["correct"] and not c["correct"])
    n_discordant = b + c
    if n_discordant == 0:
        return float("nan")
    stat = (abs(b - c) - 1) ** 2 / n_discordant
    return float(stats.chi2.sf(stat, df=1))


p_a_vs_base = mcnemar_p(baseline_res, steered_a_res)
p_b_vs_base = mcnemar_p(baseline_res, steered_b_res)
p_b_vs_a = mcnemar_p(steered_a_res, steered_b_res)

summary = pd.DataFrame([{
    "model": MODEL_ID, "n_mcq_samples": N_MCQ_SAMPLES, "top_k_heads": TOP_K_HEADS,
    "top15_overlap": len(overlap),
    "baseline_acc": b_acc,
    "steered_final_query_acc": a_acc, "p_final_query_vs_base": p_a_vs_base,
    "steered_cot_trace_acc": b2_acc, "p_cot_trace_vs_base": p_b_vs_base,
    "p_cot_trace_vs_final_query": p_b_vs_a,
}])
pd.set_option("display.width", 160)
print(summary.T.to_string())

                                                    0
model                       Qwen/Qwen3-VL-8B-Thinking
n_mcq_samples                                     150
top_k_heads                                        15
top15_overlap                                       0
baseline_acc                                     0.28
steered_final_query_acc                          0.24
p_final_query_vs_base                        0.450982
steered_cot_trace_acc                        0.713333
p_cot_trace_vs_base                               0.0
p_cot_trace_vs_final_query                        0.0


## Result

(filled in after running)